# 04 - Supervised Model Selection & Calibration

Three tabular models are compared by **PR-AUC on validation** (not accuracy):
Logistic Regression, Random Forest, LightGBM. Probabilities are then
calibrated (Platt vs Isotonic, chosen by Brier).


In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


In [ ]:
import pandas as pd, numpy as np, os, json
from src.config import get_settings
from src.supervised_model import train_and_compare
from src.feature_engineering import MODEL_FEATURES
from src.calibration import fit_best

cfg = get_settings()
base = os.path.join(os.getcwd(), "data", "processed")
tr = pd.read_parquet(os.path.join(base, "train_features.parquet"))
va = pd.read_parquet(os.path.join(base, "val_features.parquet"))
Xtr, ytr = tr[MODEL_FEATURES], tr['is_fraud'].to_numpy()
Xva, yva = va[MODEL_FEATURES], va['is_fraud'].to_numpy()


In [ ]:
results = train_and_compare(Xtr, ytr, Xva, yva, cfg)
best = results['__best__']
for k, v in results.items():
    if k != '__best__':
        print(f"  {k:24s} PR-AUC={v['pr_auc_val']:.4f}  "
              f"ROC-AUC={v['roc_auc_val']:.4f}")
print("primary:", best)


In [ ]:
model = results[best]['model']
raw_val = model.predict_proba(Xva)[:, 1]
cal, cal_rep = fit_best(np.asarray(raw_val), yva, cfg)
print("selected calibration:", cal_rep['selected'],
      "brier:", cal_rep['selected_brier'])
print(json.dumps(cal_rep['methods'], indent=2))


In [ ]:
from src.calibration import calibration_curve_data
p = cal.predict(np.asarray(raw_val))
cd = calibration_curve_data(yva, p, bins=10)
pd.DataFrame(cd).plot(figsize=(7, 4), title="reliability diagram");
print("ECE:", cd['ece'])


## Takeaway
Calibrated `fraud_probability` is the signal the risk engine consumes. On the
validation fold the primary model reaches PR-AUC ≈ 0.89 with ECE well under
1%, so the probability scale is trustworthy for thresholding.
